# XoL Tower Example

This notebook generates frequency-severity losses and applies an excess-of-loss tower using PAL's module-oriented API.

In [ ]:
import numpy as np

from pal import config, contracts, distributions, frequency_severity

config.n_sims = 100_000

severity = distributions.GPD(shape=0.33, scale=100_000, loc=1_000_000)
frequency = distributions.Poisson(mean=2)
losses = frequency_severity.FrequencySeverityModel(frequency, severity).generate()
losses = np.minimum(losses, 5_000_000) * 1.05
inflation = distributions.Normal(0.05, 0.02).generate()
gross_losses = losses * (1 + inflation)

In [ ]:
tower = contracts.XoLTower(
    limit=[1_000_000, 1_000_000, 1_000_000, 1_000_000, 10_000_000],
    excess=[1_000_000, 2_000_000, 3_000_000, 4_000_000, 5_000_000],
    aggregate_limit=[3_000_000, 2_000_000, 1_000_000, 1_000_000, 10_000_000],
    premium=[5_000, 4_000, 3_000, 2_000, 1_000],
    reinstatement_cost=[[1, 1, 1]] * 5,
)
result = tower.apply(gross_losses)
result.recoveries.aggregate().cdf_plot("Recoveries")